<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 36px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; National Coverage Extension</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        Recover the <strong>national-only German applicants PATSTAT can&rsquo;t see</strong> &mdash; from the DPMA register, mapped to the <strong>same NUTS regions</strong>.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; <a href="https://patentreports.depa.tech" target="_blank"
           style="color: #be0f05; text-decoration: none; font-weight: 600;">created by Arne Kr&uuml;ger</a>
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 620px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What you will do in this notebook</strong>
            <br/>Part&nbsp;1 &nbsp;&middot;&nbsp; <strong>Offline demo</strong> &mdash; parse bundled register records &amp; attach NUTS
            <br/>Part&nbsp;2 &nbsp;&middot;&nbsp; <strong>Live fetch</strong> from DPMAconnect&nbsp;Plus (optional)
            <br/>Part&nbsp;3 &nbsp;&middot;&nbsp; <strong>Aggregate</strong> applicant records by NUTS region
            <br/>Caveats &nbsp;&middot;&nbsp; know what this can &amp; cannot see
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 620px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Run the cells top to bottom.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            Part&nbsp;1 runs <strong>offline out of the box</strong> &mdash; no credentials, no network.
            For the live fetch in Part&nbsp;2, set <code>DPMA_USER</code> / <code>DPMA_PASS</code> in the environment;
            it degrades gracefully if unavailable. This is an <strong>optional extension</strong> to the
            <em>Regional Analysis for Lead Generation</em> notebook &mdash; it uses the DPMA register, not PATSTAT.
        </div>
    </div>
    <div style="margin-top: 20px; font-size: 12px; color: #cbd5e1;">
        Part of EPO TIP Working Group Sessions, 2026. &nbsp;Data: DPMAconnect&nbsp;Plus register &nbsp;&middot;&nbsp; Eurostat GISCO PLZ&rarr;NUTS (NUTS&nbsp;2024).
    </div>
</div>

## How this complements the PATSTAT notebook

Same goal as the first workshop notebook — **turn patent activity in a region into a lead list
of local applicants** — but sourced from the **DPMA national register** instead of PATSTAT.
Why: PATSTAT geocodes the EP/PCT route only, so ~70 % of German national filings carry **no
NUTS** and are invisible to a region filter. The national register has every filing's applicant
**address incl. PLZ**, so it recovers exactly that missing tail.

**The entry point is a time period, not a name.** We pull *all* filings registered in a period,
read their applicant addresses, map PLZ → NUTS region, then filter/aggregate by region:

```
pick a period ──▶ pull ALL register records ──▶ applicant + PLZ ──▶ NUTS3 / Bundesland
                                                                          │
                                          filter to a region ──▶ deduped lead list
```

Bulk route: **`getRegisterabzuege/<date>/<period>`** → a ZIP with one ST.36 record per
registration. (The sibling `getPublikationsdaten_XML` route needs a separate DPMAconnect
permission the workshop account may not have.) Helpers live in [`dpma/`](dpma/README.md);
design in [`docs/national-coverage-extension-dpmaconnect.md`](docs/national-coverage-extension-dpmaconnect.md).

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Make the `dpma` helper package importable regardless of where Jupyter launched.
ROOT = next(c for c in [Path.cwd(), *Path.cwd().parents]
            if (c / "dpma" / "register_parser.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dpma import (iter_registrations_from_zip, applicant_rows, enrich_rows, fetch)

SAMPLES = ROOT / "dpma" / "samples"
pd.set_option("display.max_rows", 60)
print("helpers loaded from", ROOT / "dpma")

helpers loaded from /Users/arnekrueger/Development/mtcberlin/epo-tip4patlibs/5_lead_generation/dpma


In [2]:
# --- DPMA credentials from a local .env file (never commit .env) -------------
# Put your DPMA_USER=... and DPMA_PASS=... into a .env file (in the repo root or
# this folder). This loads them into the environment so fetch.DpmaClient() finds
# them. Stdlib only -- no python-dotenv needed.
import os
from pathlib import Path

def load_env(filename=".env"):
    """Load KEY=VALUE lines from the nearest .env (searching upwards) into os.environ."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        f = base / filename
        if f.exists():
            for line in f.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, val = line.split("=", 1)
                os.environ.setdefault(key.strip(), val.strip().strip('"').strip("'"))
            return f
    return None

_envfile = load_env()
print("Loaded .env from:", _envfile if _envfile else "(no .env found)")
print("DPMA_USER set:", bool(os.environ.get("DPMA_USER")),
      "| DPMA_PASS set:", bool(os.environ.get("DPMA_PASS")))

Loaded .env from: /Users/arnekrueger/Development/mtcberlin/epo-tip4patlibs/.env
DPMA_USER set: True | DPMA_PASS set: True


## A reusable lead-list builder

Parse a register-extract ZIP into one row per applicant, attach the NUTS region, and (for the
lead list) collapse to one row per company with its filing count, IP-right kinds and a sample
IPC. Foreign applicants have no German PLZ, so they are dropped from the regional view.

In [3]:
def extract_to_rows(zip_bytes):
    """Register-extract ZIP -> enriched applicant rows (one per applicant)."""
    rows = []
    for reg in iter_registrations_from_zip(zip_bytes):
        rows += applicant_rows(reg)
    enrich_rows(rows)                      # adds nuts3 / nuts1 / bundesland
    return pd.DataFrame(rows)


def leads_for_region(df, region=None, level="bundesland"):
    """Deduped company lead list for a region (or all of DE if region is None)."""
    de = df[df["country"] == "DE"].dropna(subset=["nuts3"]).copy()
    if region is not None:
        de = de[de[level] == region]
    # isinstance guards keep missing values (some records have no kind/IPC) out of
    # the sort — a bare `if x` would let a NaN through and break sorted().
    g = (de.groupby(["name", "plz", "city", "nuts3", "bundesland"],
                    dropna=False, sort=False)
           .agg(filings=("appln_number", "nunique"),
                kinds=("kind", lambda s: "/".join(sorted({x for x in s if isinstance(x, str) and x}))),
                sample_ipc=("ipc", lambda s: next((x for x in s if isinstance(x, str) and x), "")))
           .reset_index()
           .sort_values("filings", ascending=False))
    return g

## Part 1 — Offline demo (bundled mini-extract)

Runs without credentials. `dpma/samples/registerabzug_sample_2026-06-20_weekly.zip` is a
curated 27-record slice of a real weekly extract — **company applicants only, inventor
personal data removed** (GDPR). Real weekly extracts hold ~6 000–7 000 records; use Part 2
for those.

In [4]:
sample_zip = (SAMPLES / "registerabzug_sample_2026-06-20_weekly.zip").read_bytes()
df = extract_to_rows(sample_zip)

# Which regions is this week's activity coming from?
by_region = (df[df["country"] == "DE"].dropna(subset=["nuts3"])
               .groupby("bundesland").size()
               .sort_values(ascending=False).rename("applicant_records"))
by_region.to_frame()

,applicant_records
bundesland,
Baden-Württemberg,3
Bayern,3
Hessen,3
Niedersachsen,3
Nordrhein-Westfalen,3
Sachsen-Anhalt,3
Schleswig-Holstein,3
Thüringen,3
Berlin,1


Pick a region and get the deduped lead list — the local companies filing that period, exactly
the contacts a PATLIB wants (many of them national-only, so PATSTAT never lists them).

In [5]:
REGION = "Thüringen"     # any Bundesland from the table above
leads_for_region(df, REGION)

,name,plz,city,nuts3,bundesland,filings,kinds,sample_ipc
1,Carl Zeiss Microscopy GmbH,07745,Jena,DEG03,Thüringen,2,A,G02B 21/00; G02B 21/02
0,Schliess- und Sicherungssysteme GmbH,99974,Mühlhausen,DEG09,Thüringen,1,A,E05B 63/08


## Part 2 — Live population pull (optional)

Needs `DPMA_USER` / `DPMA_PASS` and egress to `dpmaconnect.dpma.de`. Pull a **whole**
weekly extract and build the regional picture from it. A weekly extract is ~12 MB /
6 000–7 000 records and includes foreign applicants and EP validations — we keep the DE
subset. Parsing the whole week takes ~10–20 s. Degrades gracefully if unavailable.

> `date` must be a real extract date (`YYYY-MM-DD`); `period` ∈ `daily/weekly/monthly/yearly`.
> If you get *\"no records\"*, try an adjacent date.

In [6]:
EXTRACT_DATE = "2026-06-20"    # a known-good weekly extract date; adjust as needed
PERIOD = "weekly"

live_df = None
try:
    client = fetch.DpmaClient()                      # reads DPMA_USER / DPMA_PASS
    zip_bytes = client.get_register_extract(EXTRACT_DATE, PERIOD)
    live_df = extract_to_rows(zip_bytes)
    de = live_df[live_df["country"] == "DE"].dropna(subset=["nuts3"])
    print(f"{len(live_df)} applicant records total | {len(de)} German (region-mapped) | "
          f"{len(live_df) - len(de)} foreign/unmapped")
    display(de.groupby("bundesland").size()
              .sort_values(ascending=False).rename("applicant_records").to_frame())
except Exception as e:
    print("Live pull skipped:", type(e).__name__, "-", e)

6429 applicant records total | 1953 German (region-mapped) | 4476 foreign/unmapped


,applicant_records
bundesland,
Baden-Württemberg,700
Bayern,435
Nordrhein-Westfalen,327
Niedersachsen,108
Hessen,74
Sachsen,63
Thüringen,55
Rheinland-Pfalz,46
Schleswig-Holstein,37


Turn the live population into a lead list for one region:

In [8]:
if live_df is not None:
    TARGET = "Thüringen"
    display(leads_for_region(live_df, TARGET).head(40))
else:
    print("Run the live cell above first (needs credentials).")

,name,plz,city,nuts3,bundesland,filings,kinds,sample_ipc
2,Carl Zeiss Microscopy GmbH,07745,Jena,DEG03,Thüringen,17,A,G02B 21/00; G02B 21/02
19,Lübbers Anlagen- und Umwelttechnik GmbH,99947,Bad Langensalza,DEG09,Thüringen,6,A/U,F26B 23/02
5,Carl Zeiss Meditec AG,07745,Jena,DEG03,Thüringen,3,A,A61F 9/007; G02B 30/00; G02B 21/00
7,j-fiber GmbH,07751,Jena,DEG03,Thüringen,3,A,G02B 6/44; G02F 2/02; G02B 6/46; G02B 6/02; G0...
12,Glatt Ingenieurtechnik Gesellschaft mit beschr...,99427,Weimar,DEG05,Thüringen,2,A,C05B 5/00; C01B 25/26; B01J 2/00
1,Schliess- und Sicherungssysteme GmbH,99974,Mühlhausen,DEG09,Thüringen,2,A,E05B 63/08
0,"Schmidt, Werner",07819,Triptis,DEG0K,Thüringen,1,A,B62D 47/00; B62D 61/04; B62K 17/00; B62D 63/00...
17,X-FAB Global Services GmbH,99097,Erfurt,DEG01,Thüringen,1,A,H10F 30/221; H10F 71/00; H10D 62/834; H10D 62/...
26,Bauerfeind AG,07937,Zeulenroda-Triebes,DEG0L,Thüringen,1,A,A61F 5/02; A61F 5/03; A41C 1/10
25,Hampe & Schellhorn Service GbR,99310,Arnstadt,DEG0T,Thüringen,1,A,H02S 50/00


## Caveats

- **What the period contains:** `getRegisterabzuege` is the register-change stream — records
  with register activity in that period, incl. older applications with a new legal-status
  event and many foreign applicants / EP validations. Keep the DE subset for regional leads;
  filter on `filing_date` if you want only genuinely new filings.
- **Dedupe:** the same company can appear across records and periods — `leads_for_region`
  collapses by `(name, plz)`. Name spelling varies (\"GmbH\" vs \"Gesellschaft mit …\"); a
  fuzzy/harmonised applicant name would tighten this further.
- **Foreign applicants** carry no German PLZ → dropped from the regional view by design.
- **GDPR:** company addresses are low-risk B2B; natural-person applicant addresses are
  personal data — handle accordingly. The bundled sample has inventor data removed.
- **NUTS vintage:** regions use NUTS 2024 (see [`docs/nuts-mappings.md`](docs/nuts-mappings.md));
  align with your PATSTAT edition before a strict code-level join.
- **Next step:** link each firm to its PATSTAT families (Aktenzeichen ↔ `appln_nr`) with a
  `has_EP` flag to separate national-only firms from EP/PCT-active ones.